In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

import os
import json
import pickle
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACTS_DIR = Path("../../model_training/models_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
candidates = [
    Path("Data/processed/master_dataset.csv"),   # if cwd = root
    Path("../../Data/processed/master_dataset.csv"),     # if cwd = Data/notebooks
]

path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError("Error: no in Data/processed/data.csv")

df = pd.read_csv(path)

id_col = 'APD ID'
seq_col = 'Sequence'
len_col = 'Length'
condition_cols = [
    'is_antibacterial', 'is_anti_gram_positive', 'is_anti_gram_negative',
    'is_antifungal', 'is_antiviral', 'is_antiparasitic', 'is_anticancer'
]

existing_cols = [col for col in [id_col, seq_col, len_col] + condition_cols if col in df.columns]
df = df[existing_cols].dropna(subset=[seq_col])

canonical_aas = set("ACDEFGHIKLMNPQRSTVWY")

df[seq_col] = df[seq_col].astype(str).str.upper()
df = df[df[seq_col].map(lambda s: all(ch in canonical_aas for ch in s))]
df = df.drop_duplicates(subset=[seq_col]).reset_index(drop=True)

for col in condition_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

df["Length"] = df["Length"].astype("Int64")
print(f"Size after cleaning: {df.shape}")
df.head()

Size after cleaning: (6228, 10)


,APD ID,Sequence,Length,is_antibacterial,is_anti_gram_positive,is_anti_gram_negative,is_antifungal,is_antiviral,is_antiparasitic,is_anticancer
0,AP05452,AAAGKHKNKGKKNGKHNGWK,20,1,1,1,1,0,0,0
1,AP05433,AAAHKHGHGHGKHKNKGKKN,20,1,1,1,1,0,0,0
2,AP01515,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,40,1,1,1,0,0,0,0
3,AP00612,AAEFPDFYDSEEQMGPHQEAEDEKDRADQRVLTEEEKKELENLAAM...,60,1,1,1,0,0,0,0
4,AP01914,AAFRGCWTKNYSPKPCL,17,1,1,0,0,0,0,0


In [4]:
all_chars = set()
for seq in df[seq_col]:
    all_chars.update(seq)

special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
vocab_list = special_tokens + sorted(all_chars)
char2idx = {ch: i for i, ch in enumerate(vocab_list)}
idx2char = {i: ch for ch, i in char2idx.items()}
vocab_size = len(vocab_list)

print(f"Vocab size: {vocab_size}")
print("Words:", list(all_chars)[:10])

Vocab size: 24
Words: ['V', 'R', 'L', 'G', 'T', 'K', 'D', 'Y', 'H', 'I']


In [5]:
def tokenize(seq, char2idx, max_len, add_sos_eos=True):
    tokens = [char2idx.get(ch, char2idx['<UNK>']) for ch in seq]
    if add_sos_eos:
        tokens = [char2idx['<SOS>']] + tokens + [char2idx['<EOS>']]
    if len(tokens) < max_len:
        tokens += [char2idx['<PAD>']] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return tokens

max_len = df[len_col].max() + 2

tokenized_seqs = []
for seq in df[seq_col]:
    tokenized = tokenize(seq, char2idx, max_len, add_sos_eos=True)
    tokenized_seqs.append(tokenized)

conditions = df[condition_cols].values.astype(np.float32)

real_lengths = np.array([len(seq) + 2 for seq in df[seq_col]])
real_lengths = np.minimum(real_lengths, max_len)

print(f"Max len after pad: {real_lengths}")

Max len after pad: [22 22 42 ... 25 28 22]


In [6]:
class AMPDataset(Dataset):
    def __init__(self, sequences, lengths, conditions):
        self.sequences = torch.LongTensor(sequences)
        self.lengths = torch.LongTensor(lengths)
        self.conditions = torch.FloatTensor(conditions)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.lengths[idx], self.conditions[idx]

dataset = AMPDataset(tokenized_seqs, real_lengths, conditions)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

with open(ARTIFACTS_DIR / "split_indices.pkl", "wb") as f:
    pickle.dump({
        "train_idx": train_dataset.indices,
        "val_idx": val_dataset.indices,
        "test_idx": test_dataset.indices
    }, f)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 4982, Val: 622, Test: 624


In [7]:
# cVAE architecture

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=char2idx['<PAD>'])
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=False, dropout=dropout)
        self.fc_mu = nn.Linear(hidden_dim + cond_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim + cond_dim, latent_dim)

    def forward(self, x, lengths, cond):
        # x: (batch, seq_len)
        # print("x.shape:", x.shape)
        # print("lengths.max():", int(lengths.max()), "x.size(1):", x.size(1))
        # assert int(lengths.max()) <= x.size(1)
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)  # hidden: (1, batch, hidden_dim)
        hidden = hidden.squeeze(0)    # (batch, hidden_dim)
        hidden_cond = torch.cat([hidden, cond], dim=1)  # (batch, hidden_dim+cond_dim)
        mu = self.fc_mu(hidden_cond)
        logvar = self.fc_logvar(hidden_cond)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=char2idx['<PAD>'])
        self.input_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embed_dim + latent_dim + cond_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(latent_dim + cond_dim, hidden_dim)

    def forward(self, x, lengths, z, cond):
        batch_size, seq_len = x.shape

        z_cond = torch.cat([z, cond], dim=1)                   # (B, latent+cond)
        h0 = self.init_h(z_cond).unsqueeze(0)                  # (1, B, H)

        embedded = self.embedding(x)                           # (B, T, E)
        embedded = self.input_dropout(embedded)

        z_cond_rep = z_cond.unsqueeze(1).repeat(1, seq_len, 1)  # (B, T, latent+cond)
        decoder_input = torch.cat([embedded, z_cond_rep], dim=-1)

        outputs, _ = self.gru(decoder_input, h0)
        logits = self.fc_out(outputs)
        return logits

class CVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, src, lengths, cond):
        mu, logvar = self.encoder(src, lengths, cond)
        z = self.reparameterize(mu, logvar)
        decoder_input = src[:, :-1]  # (batch, max_len-1)
        target = src[:, 1:]          # (batch, max_len-1) – то, что должны предсказать
        logits = self.decoder(decoder_input, lengths - 1, z, cond)
        return logits, target, mu, logvar

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

embed_dim = 64
hidden_dim = 128
latent_dim = 32
cond_dim = len(condition_cols)
dropout = 0.2
lr = 1e-3

model = CVAE(vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss(ignore_index=char2idx['<PAD>'])

def loss_function(logits, target, mu, logvar, beta=1.0):
    # logits: (batch, seq_len, vocab)
    # target: (batch, seq_len)
    recon_loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss /= target.size(0)  # усредняем по батчу
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

Using device: cpu


C:\Users\artem\Study\PMLaDL\.venv\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [9]:
import os
output_path = "../../model_training/models_artifacts"
def train_epoch(model, loader, optimizer, beta=1.0):
    model.train()
    total_loss = 0
    for src, lengths, cond in loader:
        src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
        optimizer.zero_grad()
        logits, target, mu, logvar = model(src, lengths, cond)
        loss, recon, kl = loss_function(logits, target, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, beta=1.0):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, lengths, cond in loader:
            src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
            logits, target, mu, logvar = model(src, lengths, cond)
            loss, _, _ = loss_function(logits, target, mu, logvar, beta)
            total_loss += loss.item()
    return total_loss / len(loader)

num_epochs = 100
best_val_loss = float('inf')
patience = 8
counter = 0

train_losses, val_losses = [], []
train_recon_losses, val_recon_losses = [], []
train_kl_losses, val_kl_losses = [], []

for epoch in range(num_epochs):
    beta = min(1.0, epoch / 10)  # warmup for KL

    model.train()
    total_train, total_recon, total_kl = 0, 0, 0
    for src, lengths, cond in train_loader:
        src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
        optimizer.zero_grad()

        logits, target, mu, logvar = model(src, lengths, cond)
        loss, recon, kl = loss_function(logits, target, mu, logvar, beta=beta)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()

    train_loss = total_train / len(train_loader)
    train_recon = total_recon / len(train_loader)
    train_kl = total_kl / len(train_loader)

    model.eval()
    total_val, total_val_recon, total_val_kl = 0, 0, 0
    with torch.no_grad():
        for src, lengths, cond in val_loader:
            src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
            logits, target, mu, logvar = model(src, lengths, cond)
            loss, recon, kl = loss_function(logits, target, mu, logvar, beta=beta)

            total_val += loss.item()
            total_val_recon += recon.item()
            total_val_kl += kl.item()

    val_loss = total_val / len(val_loader)
    val_recon = total_val_recon / len(val_loader)
    val_kl = total_val_kl / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_recon_losses.append(train_recon)
    val_recon_losses.append(val_recon)
    train_kl_losses.append(train_kl)
    val_kl_losses.append(val_kl)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"beta={beta:.2f} | "
        f"train={train_loss:.4f} (recon={train_recon:.4f}, kl={train_kl:.4f}) | "
        f"val={val_loss:.4f} (recon={val_recon:.4f}, kl={val_kl:.4f})"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), ARTIFACTS_DIR / "best_cvae.pt")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(ARTIFACTS_DIR / "best_cvae.pt", map_location=device))

Epoch 1/100 | beta=0.00 | train=2.9449 (recon=2.9449, kl=12.9555) | val=2.7892 (recon=2.7892, kl=32.7273)
Epoch 2/100 | beta=0.10 | train=3.6030 (recon=2.8095, kl=7.9347) | val=2.8241 (recon=2.7864, kl=0.3768)
Epoch 3/100 | beta=0.20 | train=2.7995 (recon=2.7629, kl=0.1828) | val=2.7367 (recon=2.7180, kl=0.0937)
Epoch 4/100 | beta=0.30 | train=2.7331 (recon=2.7146, kl=0.0616) | val=2.6922 (recon=2.6800, kl=0.0405)
Epoch 5/100 | beta=0.40 | train=2.6917 (recon=2.6800, kl=0.0293) | val=2.6603 (recon=2.6520, kl=0.0207)
Epoch 6/100 | beta=0.50 | train=2.6578 (recon=2.6499, kl=0.0157) | val=2.6328 (recon=2.6270, kl=0.0115)
Epoch 7/100 | beta=0.60 | train=2.6297 (recon=2.6244, kl=0.0089) | val=2.6009 (recon=2.5968, kl=0.0068)
Epoch 8/100 | beta=0.70 | train=2.5974 (recon=2.5935, kl=0.0055) | val=2.5652 (recon=2.5621, kl=0.0043)
Epoch 9/100 | beta=0.80 | train=2.5607 (recon=2.5578, kl=0.0036) | val=2.5326 (recon=2.5302, kl=0.0030)
Epoch 10/100 | beta=0.90 | train=2.5287 (recon=2.5264, kl=0.00

<All keys matched successfully>

In [10]:
def generate(model, cond, max_gen_len=None, temperature=1.0):
    if max_gen_len is None:
        max_gen_len = max_len

    model.eval()
    with torch.no_grad():
        cond = torch.FloatTensor(cond).unsqueeze(0).to(device)
        z = torch.randn(1, latent_dim).to(device)
        z_cond = torch.cat([z, cond], dim=1)
        h = model.decoder.init_h(z_cond).unsqueeze(0)

        input_token = torch.LongTensor([[char2idx['<SOS>']]]).to(device)
        generated = []

        for _ in range(max_gen_len):
            embedded = model.decoder.embedding(input_token)
            z_step = z_cond.unsqueeze(1)
            decoder_input = torch.cat([embedded, z_step], dim=-1)

            output, h = model.decoder.gru(decoder_input, h)
            logits = model.decoder.fc_out(output.squeeze(1))
            probs = torch.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

            if next_token == char2idx['<EOS>']:
                break

            generated.append(next_token)
            input_token = torch.LongTensor([[next_token]]).to(device)

        seq = ''.join(
            idx2char[idx] for idx in generated
            if idx not in [char2idx['<PAD>'], char2idx['<SOS>'], char2idx['<EOS>']]
        )
        return seq

cond_gram_pos = [0,1,0,0,0,0,0]
for i in range(5):
    print(generate(model, cond_gram_pos, temperature=0.9))

GLFSSLKGVATGVAKKAGKNALKGVASKVLNAMTCKLQGQC
GIGCGTSEFCCPNALFGRCGTHCSCVFAGCKVGR
FLPVLVGAVGSVLPSVVCSISKVC
NGALCGTEGVRLRHGICKLTH
FLSLIPSAARLAKKIF


In [11]:
with open(ARTIFACTS_DIR / "vocab.pkl", "wb") as f:
    pickle.dump({"char2idx": char2idx, "idx2char": idx2char, "max_len": max_len}, f)

with open(ARTIFACTS_DIR / "cvae_config.json", "w") as f:
    json.dump({
        "embed_dim": embed_dim,
        "hidden_dim": hidden_dim,
        "latent_dim": latent_dim,
        "cond_dim": cond_dim,
        "dropout": dropout,
        "lr": lr,
        "seed": SEED,
        "condition_cols": condition_cols,
        "seq_col": seq_col,
        "len_col": len_col
    }, f, indent=2)

with open(ARTIFACTS_DIR / "training_history.pkl", "wb") as f:
    pickle.dump({
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_recon_losses": train_recon_losses,
        "val_recon_losses": val_recon_losses,
        "train_kl_losses": train_kl_losses,
        "val_kl_losses": val_kl_losses
    }, f)